In [ ]:
import os
from pathlib import Path
from string import ascii_lowercase

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

if "MPL_STYLE" not in os.environ:
    os.environ["MPL_STYLE"] = "seaborn-v0_8-notebook"
plt.style.use(os.environ["MPL_STYLE"])


In [ ]:
OUTPUT_BASE = Path(os.getenv("OUTPUT_BASE")).resolve(strict=True)

SUMMARY_DIR = OUTPUT_BASE / "summary" / "M6"
SUMMARY_CSV_PATH = (SUMMARY_DIR / "summary.csv").resolve(strict=True)

summary_df = (
    pd.read_csv(SUMMARY_CSV_PATH)
    .sort_values(
        by=[
            "target_label",
            "model_family",
            "feature_set_label",
            "model_label",
        ]
    )
    .reset_index(drop=True)
)
print(summary_df.columns.tolist())
summary_df.head(n=10)

In [ ]:
summary_df["model_family"].unique()

In [ ]:
MODEL_FAMILY = {
    "summary_stats": {
        "label": "Summary Stats",
        "label_short": "SS",
        "color": "#1f77b4",
        "marker": "o",
    },
    "deep_sets": {
        "label": "Deep Sets",
        "label_short": "DS",
        "color": "#2ca02c",
        "marker": "s",
    },
    "set_transformer": {
        "label": "Set Transformer",
        "label_short": "ST",
        "color": "#ff7f0e",
        "marker": "^",
    },
}

FEATURE_SET_PLOT_CONFIG = {
    "sky": {"label": "Sky", "color": "#1f77b4"},
    "sky+L": {"label": "Sky$+L$", "color": "#6baed6"},
    "cartesian": {"label": "Cartesian", "color": "#d62728"},
    "cartesian+L": {"label": "Cartesian$+L$", "color": "#fb6a4a"},
}

TARGET_PLOT_CONFIG = {
    "age": {"label": "Age", "unit": "Myr"},
    "total_mass": {"label": "Total Mass", "unit": "$M_\\odot$"},
}

## (Sample-Balanced) MAE Comparison Across Model Families

In [ ]:
fig, (age_ax, phantom_ax, mass_ax, legend_ax) = plt.subplots(
    nrows=1,
    ncols=4,
    figsize=(17, 5),
    gridspec_kw=dict(width_ratios=[6, 1, 6, 1], wspace=0),
    dpi=300,
)
phantom_ax.axis("off")
legend_ax.axis("off")

n_families = len(MODEL_FAMILY)
n_features = len(FEATURE_SET_PLOT_CONFIG)
feature_step = 0.8 / (n_features - 1)
family_spacing = 1.5
family_centers = np.arange(n_families) * family_spacing
family_boundaries = (family_centers[:-1] + family_centers[1:]) / 2
offsets = (np.arange(n_features) - (n_features - 1) / 2) * feature_step


def _marker_area(param_count):
    return np.interp(
        np.log10(param_count),
        [
            np.log10(summary_df["model_param_count"].min()),
            np.log10(summary_df["model_param_count"].max()),
        ],
        [24, 120],
    )


feature_legend_handles = {
    fs_config["label"]: mpl.lines.Line2D(
        [],
        [],
        linestyle="none",
        marker="o",
        markersize=10,
        markerfacecolor=fs_config["color"],
        markeredgecolor="none",
    )
    for fs_config in FEATURE_SET_PLOT_CONFIG.values()
}

size_ref = np.array([50, 200, 500, 1000], dtype=int)
size_legend_handles = [
    mpl.lines.Line2D(
        [],
        [],
        linestyle="none",
        marker="o",
        markersize=np.sqrt(_marker_area(count)),
        markerfacecolor="white",
        markeredgecolor="k",
        markeredgewidth=0.5,
    )
    for count in size_ref
]

for ax_idx, (ax, target_label) in enumerate(
    [
        (age_ax, "age"),
        (mass_ax, "total_mass"),
    ]
):
    ax.text(
        0.95,
        0.95,
        f"({ascii_lowercase[ax_idx]})",
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=20,
    )
    config = TARGET_PLOT_CONFIG[target_label]
    ax.set_title(f"{config['label']} [{config['unit']}]", fontsize=20)

    target_df = summary_df[summary_df["target_label"] == target_label]

    for x_idx, model_family in enumerate(MODEL_FAMILY.keys()):
        family_df = target_df[target_df["model_family"] == model_family]
        for xx_idx, (feature_set_label, fs_config) in enumerate(
            FEATURE_SET_PLOT_CONFIG.items()
        ):
            model_df = family_df[
                family_df["feature_set_label"] == feature_set_label
            ].sort_values("model_param_count", ascending=False)
            x_pos = family_centers[x_idx] + offsets[xx_idx]
            for row in model_df.itertuples(index=False):
                ax.scatter(
                    x_pos,
                    row.mae,
                    color=fs_config["color"],
                    alpha=0.9,
                    lw=0.3,
                    ec="k",
                    s=_marker_area(row.model_param_count),
                    zorder=3,
                )

    ax.set_xlabel("Model Family")
    ax.set_xticks(family_centers)
    ax.set_xticklabels(
        list([model_cfg["label_short"] for model_cfg in MODEL_FAMILY.values()]),
        ha="center",
    )
    ax.set_xlim(family_centers[0] - 0.65, family_centers[-1] + 0.65)
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.set_ylabel("MAE")

    for boundary in family_boundaries:
        ax.axvline(boundary, color="k", lw=1.2, ls="--", zorder=1)

    if ax is age_ax:
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(20))
        ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(5))
        ax.set_ylim(10, 50)
    else:
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(9, offset=3))
        ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(3))
        ax.set_ylim(0, 27)

legend_ax.legend(
    [
        mpl.lines.Line2D([], [], linestyle="none"),
        *feature_legend_handles.values(),
        mpl.lines.Line2D([], [], linestyle="none"),
        *size_legend_handles,
    ],
    [
        "Feature Set",
        *feature_legend_handles.keys(),
        "Model Params",
        *map(lambda value: f"${value:,}$".replace(",", r"\,"), size_ref),
    ],
    loc="center left",
    frameon=True,
    handlelength=1.0,
    handletextpad=0.8,
    labelspacing=0.75,
    borderpad=0.8,
)
fig.savefig(SUMMARY_DIR / "mae_by_model_family.pdf", bbox_inches="tight")
plt.show()


In [ ]:
fig, (age_ax, phantom_ax, mass_ax, legend_ax) = plt.subplots(
    nrows=1,
    ncols=4,
    figsize=(17, 5),
    gridspec_kw=dict(width_ratios=[6, 1, 6, 1], wspace=0),
    dpi=300,
)
phantom_ax.axis("off")
legend_ax.axis("off")

n_families = len(MODEL_FAMILY)
n_features = len(FEATURE_SET_PLOT_CONFIG)
feature_step = 0.8 / (n_features - 1)
family_spacing = 1.5
family_centers = np.arange(n_families) * family_spacing
family_boundaries = (family_centers[:-1] + family_centers[1:]) / 2
offsets = (np.arange(n_features) - (n_features - 1) / 2) * feature_step


def _marker_area(param_count):
    return np.interp(
        np.log10(param_count),
        [
            np.log10(summary_df["model_param_count"].min()),
            np.log10(summary_df["model_param_count"].max()),
        ],
        [24, 120],
    )


feature_legend_handles = {
    fs_config["label"]: mpl.lines.Line2D(
        [],
        [],
        linestyle="none",
        marker="o",
        markersize=10,
        markerfacecolor=fs_config["color"],
        markeredgecolor="none",
    )
    for fs_config in FEATURE_SET_PLOT_CONFIG.values()
}

size_ref = np.array([50, 200, 500, 1000], dtype=int)
size_legend_handles = [
    mpl.lines.Line2D(
        [],
        [],
        linestyle="none",
        marker="o",
        markersize=np.sqrt(_marker_area(count)),
        markerfacecolor="white",
        markeredgecolor="k",
        markeredgewidth=0.5,
    )
    for count in size_ref
]

for ax_idx, (ax, target_label) in enumerate(
    [
        (age_ax, "age"),
        (mass_ax, "total_mass"),
    ]
):
    ax.text(
        0.95,
        0.95,
        f"({ascii_lowercase[ax_idx]})",
        transform=ax.transAxes,
        va="top",
        ha="right",
        fontsize=20,
    )
    config = TARGET_PLOT_CONFIG[target_label]
    ax.set_title(f"{config['label']} [{config['unit']}]", fontsize=20)

    target_df = summary_df[summary_df["target_label"] == target_label]

    for x_idx, model_family in enumerate(MODEL_FAMILY.keys()):
        family_df = target_df[target_df["model_family"] == model_family]
        for xx_idx, (feature_set_label, fs_config) in enumerate(
            FEATURE_SET_PLOT_CONFIG.items()
        ):
            model_df = family_df[
                family_df["feature_set_label"] == feature_set_label
            ].sort_values("model_param_count", ascending=False)
            x_pos = family_centers[x_idx] + offsets[xx_idx]
            for row in model_df.itertuples(index=False):
                ax.scatter(
                    x_pos,
                    row.balanced_mae,
                    color=fs_config["color"],
                    alpha=0.9,
                    lw=0.3,
                    ec="k",
                    s=_marker_area(row.model_param_count),
                    zorder=3,
                )

    ax.set_xlabel("Model Family")
    ax.set_xticks(family_centers)
    ax.set_xticklabels(
        list([model_cfg["label_short"] for model_cfg in MODEL_FAMILY.values()]),
        ha="center",
    )
    ax.set_xlim(family_centers[0] - 0.65, family_centers[-1] + 0.65)
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.set_ylabel("Balanced MAE")

    for boundary in family_boundaries:
        ax.axvline(boundary, color="k", lw=1.2, ls="--", zorder=1)

    if ax is age_ax:
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(9, offset=3))
        ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(3))
        ax.set_ylim(0, 36)
    else:
        ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(4, offset=2))
        ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(2))
        ax.set_ylim(0, 16)

legend_ax.legend(
    [
        mpl.lines.Line2D([], [], linestyle="none"),
        *feature_legend_handles.values(),
        mpl.lines.Line2D([], [], linestyle="none"),
        *size_legend_handles,
    ],
    [
        "Feature Set",
        *feature_legend_handles.keys(),
        "Model Params",
        *map(lambda value: f"${value:,}$".replace(",", r"\,"), size_ref),
    ],
    loc="center left",
    frameon=True,
    handlelength=1.0,
    handletextpad=0.8,
    labelspacing=0.75,
    borderpad=0.8,
)
fig.savefig(SUMMARY_DIR / "balanced_mae_by_model_family.pdf", bbox_inches="tight")
plt.show()
